# Сравнение подходов к сходству неймингов

In [1]:
PAIRS = [
    # ===================== звуковое сходство
    ("europlex", "euroflex"),
    ("тигота", "дикота"),
    ("реком", "рекод"),
    ("селексен", "cellexin"),
    ("диксика", "dixy"),
    # ===================== вхождение
    ("проби", "пробимакс"),
    ("кола", "кока кола"),
    # ===================== транслитерация
    ("arkline", "арклайн"),
    ("europlex", "европлекс"),
    ("phoenix", "феникс"),
    # ===================== перевод
    ("молоко", "milk"),
    ("ковчег", "ark"),
    # ===================== перестановка слов
    ("лоза тамани", "тамани лоза"),
    # ===================== однокоренные, НЕ сходные по смыслу
    ("лесное", "лесник"),
    ("таежник", "таежная"),
    # ===================== ложное вхождение
    ("руси", "крусис"),
    # ===================== случайная пара
    ("europlex", "тигота"),
    ("арклайн", "молоко"),
]

## 1. Текстовые / лексические метрики

In [2]:
import pandas as pd

from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler, Levenshtein


def trigram_jaccard(a: str, b: str) -> float:
    ta = {a[i : i + 3] for i in range(max(len(a) - 2, 1))}
    tb = {b[i : i + 3] for i in range(max(len(b) - 2, 1))}
    return len(ta & tb) / max(len(ta | tb), 1)


pd.DataFrame(
    [
        {
            "INPUT": f"{a} / {b}",
            "Levenshtein": round(Levenshtein.normalized_similarity(a, b), 3),
            "JaroWinkler": round(JaroWinkler.normalized_similarity(a, b), 3),
            "token_sort": round(fuzz.token_sort_ratio(a, b) / 100, 3),
            "token_set": round(fuzz.token_set_ratio(a, b) / 100, 3),
            "partial": round(fuzz.partial_ratio(a, b) / 100, 3),
            "trigram_jaccard": round(trigram_jaccard(a, b), 3),
        }
        for a, b in PAIRS
    ]
)

,INPUT,Levenshtein,JaroWinkler,token_sort,token_set,partial,trigram_jaccard
0,europlex / euroflex,0.875,0.950,0.875,0.875,0.875,0.333
1,тигота / дикота,0.667,0.778,0.667,0.667,0.727,0.143
2,реком / рекод,0.800,0.920,0.800,0.800,0.889,0.500
3,селексен / cellexin,0.000,0.000,0.000,0.000,0.000,0.000
4,диксика / dixy,0.000,0.000,0.000,0.000,0.000,0.000
5,проби / пробимакс,0.556,0.911,0.714,0.714,1.000,0.429
6,кола / кока кола,0.444,0.694,0.615,1.000,1.000,0.286
7,arkline / арклайн,0.000,0.000,0.000,0.000,0.000,0.000
8,europlex / европлекс,0.000,0.000,0.000,0.000,0.000,0.000
9,phoenix / феникс,0.000,0.000,0.000,0.000,0.000,0.000


**Выводы:**

1. Текстовые метрики лучше всего показывают себя в парах с незначительным отличием в написании.
2. Текстовые метрики плохо показывают себя на фонетических парах и переводах. Ожидаемый результат.
3. Ложное вхождение показывает высокий результат -- требуется другой подход.

## 2. Фонетические алгоритмы

| Алгоритм | Идея | Ограничение |
|---|---|---|
| Soundex (1918) | код = 1-я буква + 3 цифры классов согласных | ключ по первой букве; кириллицы нет |
| Metaphone / Double Metaphone | правила английского произношения -> код | только английская фонология |
| Beider-Morse | мультиязычные правила, несколько кодов на имя | грубая почти бинарная оценка  |
| **G2P -> IPA -> взвешенное выравнивание** | транскрипция + цены замен по артикуляционным признакам | требует G2P для ru/en |

Ниже приведены контрпримеры для алоритмов:

In [3]:
def soundex(word: str) -> str:
    word = word.upper()
    codes = {
        **{c: "1" for c in "BFPV"},
        **{c: "2" for c in "CGJKQSXZ"},
        **{c: "3" for c in "DT"},
        "L": "4",
        **{c: "5" for c in "MN"},
        "R": "6",
    }
    
    out, prev = word[0], codes.get(word[0], "")

    for ch in word[1:]:
        c = codes.get(ch, "")
        if c and c != prev:
            out += c
        if ch not in "HW":
            prev = c
    return (out + "000")[:4]


for a, b in [("TIGOTA", "DICOTA"), ("EUROPLEX", "EUROFLEX")]:
    sa, sb = soundex(a), soundex(b)
    print(f"{a:9s} -> {sa}   {b:9s} -> {sb}   match={sa == sb}")

TIGOTA    -> T230   DICOTA    -> D230   match=False
EUROPLEX  -> E614   EUROFLEX  -> E614   match=True


In [4]:
TRANSLIT = {
    "a": "а",
    "b": "б",
    "c": "к",
    "d": "д",
    "e": "е",
    "f": "ф",
    "g": "г",
    "h": "х",
    "i": "и",
    "j": "дж",
    "k": "к",
    "l": "л",
    "m": "м",
    "n": "н",
    "o": "о",
    "p": "п",
    "q": "к",
    "r": "р",
    "s": "с",
    "t": "т",
    "u": "у",
    "v": "в",
    "w": "в",
    "x": "кс",
    "y": "и",
    "z": "з",
}


def naive_translit(s):
    return "".join(TRANSLIT.get(c, c) for c in s.lower())


for a, b in [("phoenix", "феникс"), ("arkline", "арклайн")]:
    t = naive_translit(a)
    print(
        f"{a:9s} -> {t!r:12s} vs {b!r:10s} "
        f"JW={JaroWinkler.normalized_similarity(t, b):.3f}"
    )

phoenix   -> 'пхоеникс'   vs 'феникс'   JW=0.819
arkline   -> 'арклине'    vs 'арклайн'  JW=0.886


Для слов на кириллице используется модель epitran `rus-Cyrl`, которая учитывает особенности русского произношения.

Для слов на латинице рассматриваются два варианта чтения:
- английское произношение (через g2p_en, который использует словарь CMUdict и нейросеть для новых слов);
- побуквенное «континентальное» чтение.

Сходство названий рассчитывается как: 1 − взвешенное фонетическое расстояние.

Расстояние определяется сравнением фонем. Стоимость замены одной фонемы на другую зависит от того, насколько различаются их артикуляционные признаки (используются 24 признака из системы `PanPhon`).

Например, замена "т" на "д" считается небольшой, а замена гласного на согласный - серьёзной. Дополнительно замены в начале слова штрафуются сильнее, поскольку совпадение первых звуков особенно важно при восприятии названий.

## 3. Эмбеддинговые подходы

Смысловое сходство определяется по совпадению значения слов даже на разных языках. Например, "МОЛОКО" и "MILK" должны считаться похожими. При этом слова с общим корнем, но разным значением, не должны считаться сходными (например, "Лесное" и "Лесник").

Модели fastText не рассматриваются, поскольку они основаны на символьных n-граммах и поэтому часто считают однокоренные слова похожими. В данной задаче это нежелательно.

Для оценки смыслового сходства сравниваются четыре модели семейства sentence-transformers. Проверка проводится на трёх группах пар:

- переводные пары -- высокое сходство;
- однокоренные слова с разным значением -- небольшое сходство;
- случайные пары -- низкое сходство.

Качество моделей оценивается по средним значениям косинусного сходства, метрике AUC и способности находить правильные соответствия в top-k поиске

In [5]:
from pathlib import Path

import pyarrow
import pandas as pd

MODELS_DIR = Path("../models")
DEVICE = "cpu"

WEIGHT_FILES = ("model.safetensors", "pytorch_model.bin")
MODELS = {
    "rubert-tiny2": ("cointegrated/rubert-tiny2", ""),
    "LaBSE": ("sentence-transformers/LaBSE", ""),
    "e5-base": ("intfloat/multilingual-e5-base", "query: "),
    "MiniLM-L12": ("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", ""),
}

def snapshot_complete(d: Path):
    return any((d / w).exists() for w in WEIGHT_FILES) or bool(
        list(d.glob("model-*-of-*.safetensors"))
    )


for name, (hf_id, _) in MODELS.items():
    if not snapshot_complete(MODELS_DIR / name):
        from huggingface_hub import snapshot_download

        print(f"Downloading {hf_id} to {MODELS_DIR / name}")
        snapshot_download(hf_id, local_dir=MODELS_DIR / name)

print("Models:", sorted(p.name for p in MODELS_DIR.iterdir()))

Models: ['LaBSE', 'MiniLM-L12', 'e5-base', 'rubert-tiny2']


In [6]:
import torch
import torch.nn.functional as F


class SemanticEncoder:
    def __init__(self, local_path, device=None, prefix=""):
        from transformers import AutoModel, AutoTokenizer

        self.device = torch.device(device or "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(
            str(local_path), local_files_only=True
        )
        self.model = AutoModel.from_pretrained(
            str(local_path), local_files_only=True
        ).to(self.device)
        self.model.eval()
        self.prefix = prefix

    @torch.inference_mode()
    def encode(self, texts, batch_size=64, max_length=64):
        if self.prefix:
            texts = [self.prefix + t for t in texts]

        out = []

        for s in range(0, len(texts), batch_size):
            enc = self.tokenizer(
                texts[s : s + batch_size],
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            ).to(self.device)

            h = self.model(**enc).last_hidden_state
            m = enc["attention_mask"].unsqueeze(-1).float()

            pooled = (h * m).sum(1) / m.sum(1).clamp(min=1e-9)
            out.append(F.normalize(pooled, p=2, dim=1).cpu())

        return torch.cat(out)

In [7]:
RU_EN = [
    ["молоко", "milk"],
    ["ковчег", "ark"],
    ["медведь", "bear"],
    ["волк", "wolf"],
    ["лиса", "fox"],
    ["орел", "eagle"],
    ["сокол", "falcon"],
    ["лев", "lion"],
    ["тигр", "tiger"],
    ["пантера", "panther"],
    ["звезда", "star"],
    ["солнце", "sun"],
    ["луна", "moon"],
    ["небо", "sky"],
    ["облако", "cloud"],
    ["гром", "thunder"],
    ["молния", "lightning"],
    ["огонь", "fire"],
    ["пламя", "flame"],
    ["лед", "ice"],
    ["снег", "snow"],
    ["вода", "water"],
    ["река", "river"],
    ["море", "sea"],
    ["океан", "ocean"],
    ["озеро", "lake"],
    ["гора", "mountain"],
    ["лес", "forest"],
    ["дерево", "tree"],
    ["дуб", "oak"],
    ["береза", "birch"],
    ["цветок", "flower"],
    ["роза", "rose"],
    ["ромашка", "daisy"],
    ["золото", "gold"],
    ["серебро", "silver"],
    ["железо", "iron"],
    ["сталь", "steel"],
    ["камень", "stone"],
    ["алмаз", "diamond"],
    ["жемчуг", "pearl"],
    ["корона", "crown"],
    ["король", "king"],
    ["королева", "queen"],
    ["принц", "prince"],
    ["рыцарь", "knight"],
    ["замок", "castle"],
    ["башня", "tower"],
    ["мост", "bridge"],
    ["дорога", "road"],
    ["путь", "way"],
    ["линия", "line"],
    ["точка", "point"],
    ["круг", "circle"],
    ["квадрат", "square"],
    ["треугольник", "triangle"],
    ["стрела", "arrow"],
    ["щит", "shield"],
    ["меч", "sword"],
    ["ключ", "key"],
    ["дверь", "door"],
    ["окно", "window"],
    ["дом", "house"],
    ["город", "city"],
    ["деревня", "village"],
    ["сад", "garden"],
    ["поле", "field"],
    ["ферма", "farm"],
    ["урожай", "harvest"],
    ["зерно", "grain"],
    ["хлеб", "bread"],
    ["мед", "honey"],
    ["сахар", "sugar"],
    ["соль", "salt"],
    ["перец", "pepper"],
    ["вкус", "taste"],
    ["аромат", "aroma"],
    ["свежий", "fresh"],
    ["чистый", "clean"],
    ["белый", "white"],
    ["черный", "black"],
    ["красный", "red"],
    ["зеленый", "green"],
    ["синий", "blue"],
    ["желтый", "yellow"],
    ["радуга", "rainbow"],
    ["свет", "light"],
    ["тень", "shadow"],
    ["ночь", "night"],
    ["день", "day"],
    ["утро", "morning"],
    ["весна", "spring"],
    ["лето", "summer"],
    ["осень", "autumn"],
    ["зима", "winter"],
    ["время", "time"],
    ["час", "hour"],
    ["момент", "moment"],
    ["вечность", "eternity"],
    ["жизнь", "life"],
    ["здоровье", "health"],
    ["сила", "power"],
    ["энергия", "energy"],
    ["скорость", "speed"],
    ["полет", "flight"],
    ["крыло", "wing"],
    ["перо", "feather"],
    ["птица", "bird"],
    ["рыба", "fish"],
    ["кит", "whale"],
    ["дельфин", "dolphin"],
    ["акула", "shark"],
    ["пчела", "bee"],
    ["бабочка", "butterfly"],
    ["дракон", "dragon"],
    ["феникс", "phoenix"],
    ["единорог", "unicorn"],
    ["герой", "hero"],
    ["легенда", "legend"],
    ["мечта", "dream"],
    ["надежда", "hope"],
    ["вера", "faith"],
    ["любовь", "love"],
    ["радость", "joy"],
    ["счастье", "happiness"],
    ["удача", "luck"],
    ["победа", "victory"],
    ["чемпион", "champion"],
    ["лидер", "leader"],
    ["мастер", "master"],
    ["эксперт", "expert"],
    ["профессионал", "professional"],
    ["партнер", "partner"],
    ["друг", "friend"],
    ["семья", "family"],
    ["традиция", "tradition"],
    ["качество", "quality"],
    ["комфорт", "comfort"],
    ["гармония", "harmony"],
    ["баланс", "balance"],
    ["ритм", "rhythm"],
    ["мелодия", "melody"],
    ["песня", "song"],
    ["танец", "dance"],
    ["праздник", "holiday"],
    ["подарок", "gift"],
    ["сюрприз", "surprise"],
    ["секрет", "secret"],
    ["тайна", "mystery"],
    ["магия", "magic"],
    ["чудо", "miracle"],
    ["сказка", "fairytale"],
    ["история", "story"],
    ["книга", "book"],
    ["слово", "word"],
    ["буква", "letter"],
    ["знак", "sign"],
    ["символ", "symbol"],
    ["образ", "image"],
    ["зеркало", "mirror"],
    ["взгляд", "look"],
    ["глаз", "eye"],
    ["рука", "hand"],
    ["сердце", "heart"],
    ["душа", "soul"],
    ["разум", "mind"],
    ["идея", "idea"],
    ["мысль", "thought"],
    ["наука", "science"],
    ["знание", "knowledge"],
    ["школа", "school"],
    ["учитель", "teacher"],
    ["студент", "student"],
    ["доктор", "doctor"],
    ["строитель", "builder"],
    ["архитектор", "architect"],
    ["инженер", "engineer"],
    ["капитан", "captain"],
    ["пилот", "pilot"],
    ["моряк", "sailor"],
    ["якорь", "anchor"],
    ["парус", "sail"],
    ["корабль", "ship"],
    ["лодка", "boat"],
    ["порт", "port"],
    ["маяк", "lighthouse"],
    ["компас", "compass"],
    ["карта", "map"],
    ["глобус", "globe"],
    ["мир", "world"],
    ["планета", "planet"],
    ["космос", "space"],
    ["ракета", "rocket"],
    ["спутник", "satellite"],
    ["орбита", "orbit"],
    ["галактика", "galaxy"],
    ["комета", "comet"],
    ["метеор", "meteor"],
    ["горизонт", "horizon"],
    ["вершина", "peak"],
    ["вектор", "vector"],
    ["атом", "atom"],
    ["кристалл", "crystal"],
    ["формула", "formula"],
    ["код", "code"],
    ["цифра", "digit"],
    ["сеть", "network"],
    ["связь", "connection"],
    ["сигнал", "signal"],
    ["волна", "wave"],
    ["поток", "stream"],
    ["источник", "source"],
    ["начало", "start"],
    ["финиш", "finish"],
    ["центр", "center"],
    ["фокус", "focus"],
    ["импульс", "impulse"],
    ["прогресс", "progress"],
    ["будущее", "future"],
]

SAME_ROOT = [
    ("лесное", "лесник"),
    ("таежник", "таежная"),
    ("горный", "горняк"),
    ("морской", "моряк"),
    ("садовое", "садовник"),
    ("речной", "речник"),
    ("звездный", "звездочет"),
    ("печное", "печник"),
    ("водный", "водник"),
    ("дворик", "дворник"),
    ("молочко", "молочник"),
    ("снежный", "снеговик"),
]

RANDOM_PAIRS = [
    (RU_EN[i][0], RU_EN[(i + 7) % len(RU_EN)][1]) for i in range(len(RU_EN))
]


def auc(pos, neg):
    wins = ties = 0

    for p in pos:
        for n in neg:
            wins += p > n
            ties += p == n

    return (wins + 0.5 * ties) / (len(pos) * len(neg))

In [8]:
results, encoders = {}, {}
words = sorted({w for grp in (RU_EN, SAME_ROOT, RANDOM_PAIRS) for p in grp for w in p})

for name, (_, prefix) in MODELS.items():
    enc = SemanticEncoder(MODELS_DIR / name, DEVICE, prefix)
    embs = enc.encode(words, batch_size=128)
    of = {w: embs[i] for i, w in enumerate(words)}
    cos = lambda pairs: [float(of[a] @ of[b]) for a, b in pairs]
    pos, root, rnd = cos(RU_EN), cos(SAME_ROOT), cos(RANDOM_PAIRS)

    results[name] = {
        "SHAPE": embs.shape[1],
        "COSINE TRANSLATION": round(sum(pos) / len(pos), 4),
        "COSINE RELATED": round(sum(root) / len(root), 4),
        "COSINE RANDOM": round(sum(rnd) / len(rnd), 4),
        "AUC TRANSLATION/RELATED": round(auc(pos, root), 4),
        "AUC TRANSLATION/RANDOM": round(auc(pos, rnd), 4),
    }

    encoders[name] = enc

pd.DataFrame(results).T.sort_values("AUC TRANSLATION/RANDOM", ascending=False)

,SHAPE,COSINE TRANSLATION,COSINE RELATED,COSINE RANDOM,AUC TRANSLATION/RELATED,AUC TRANSLATION/RANDOM
LaBSE,768.0,0.9481,0.8312,0.6038,0.9357,1.0000
MiniLM-L12,384.0,0.9091,0.7876,0.3565,0.7747,0.9885
e5-base,768.0,0.8942,0.9081,0.8124,0.3988,0.9519
rubert-tiny2,312.0,0.8939,0.8363,0.8100,0.7740,0.8107


## 4. Top-k

Для проверки семантических моделей используется задача кросс-языкового поиска. Модель должна по русскому слову-запросу находить его английский перевод в корпусе английских слов.

В качестве корпуса используется английский лексикон, а в качестве запросов — соответствующие русские слова. Качество оценивается стандартными метриками информационного поиска:

- **hit@1** -- доля случаев, когда правильный перевод найден на первом месте;
- **hit@5** -- доля случаев, когда правильный перевод попадает в первые пять результатов;
- **MRR** -- средний показатель позиции правильного ответа в выдаче.

Для сравнения рассчитывается также лексический базовый метод: поиск по строковому сходству с использованием коэффициента **Jaro–Winkler** без учёта различий между кириллицей и латиницей. Это позволяет оценить, насколько семантическая модель превосходит простое сравнение написания слов.


In [9]:
from rapidfuzz import process


def topk_metrics(enc, k=5):
    corpus = [en for _, en in RU_EN]

    C = enc.encode(corpus, batch_size=128)
    Q = enc.encode([ru for ru, _ in RU_EN], batch_size=128)

    sims = Q @ C.T
    hit1 = hitk = mrr = 0.0

    for i in range(len(RU_EN)):
        order = sims[i].argsort(descending=True)

        rank = (order == i).nonzero()[0].item() + 1

        hit1 += rank == 1
        hitk += rank <= k

        mrr += 1 / rank

    n = len(RU_EN)

    return {
        "hit@1": round(hit1 / n, 3),
        f"hit@{k}": round(hitk / n, 3),
        "MRR": round(mrr / n, 3),
    }


def lexical_baseline(k=5):
    corpus = [en for _, en in RU_EN]
    hit1 = hitk = mrr = 0.0
    for i, (ru, _) in enumerate(RU_EN):
        scored = process.extract(
            ru, corpus, scorer=JaroWinkler.normalized_similarity, limit=len(corpus)
        )
        rank = next(j + 1 for j, (cand, _, ci) in enumerate(scored) if ci == i)
        hit1 += rank == 1
        hitk += rank <= k
        mrr += 1 / rank
    n = len(RU_EN)
    return {
        "hit@1": round(hit1 / n, 3),
        f"hit@{k}": round(hitk / n, 3),
        "MRR": round(mrr / n, 3),
    }


table = {name: topk_metrics(enc) for name, enc in encoders.items()}
table["JaroWinkler (лексический бейзлайн)"] = lexical_baseline()
pd.DataFrame(table).T.sort_values("MRR", ascending=False)

,hit@1,hit@5,MRR
LaBSE,0.977,1.000,0.987
MiniLM-L12,0.881,0.936,0.907
e5-base,0.836,0.927,0.875
rubert-tiny2,0.671,0.776,0.718
JaroWinkler (лексический бейзлайн),0.005,0.023,0.027


## 5. Выводы

1. **LaBSE выбрана как основная модель для оценки смыслового сходства.** Она лучше остальных распознаёт переводные пары слов, надёжно отличает их от случайных совпадений и от однокоренных слов с разным значением. Кроме того, модель показывает лучшие результаты при поиске соответствий.

2. **e5-base не подходит для данной задачи.** Модель часто считает однокоренные слова более похожими, чем слова-переводы. Это означает, что она ориентируется скорее на сходство формы слова, чем на сходство смысла.

3. **rubert-tiny2 также не рекомендуется использовать.** Она показывает высокое сходство даже для случайных пар слов, поэтому плохо различает действительно связанные и несвязанные понятия.

4. **Лексические методы сами по себе недостаточны.** Они практически не справляются с поиском смысловых соответствий между разными языками. Поэтому для качественной оценки необходим отдельный семантический канал, так же как фонетический канал необходим для работы с транслитерациями и различными вариантами написания.

**Итог:** для оценки сходства обозначений используются три независимых канала -- лексический, фонетический и семантический. Семантический канал реализован на базе LaBSE, поскольку именно эта модель наиболее точно выявляет совпадение значений между словами разных языков.
